# 01 — Data Exploration

Explore the Animal Image Classification Dataset (15 classes): distribution, sample images, and basic statistics.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

sns.set_style('whitegrid')
%matplotlib inline

CLASS_NAMES = [
    'bear', 'bird', 'cat', 'cow', 'deer',
    'dog', 'dolphin', 'elephant', 'giraffe', 'horse',
    'kangaroo', 'lion', 'panda', 'tiger', 'zebra'
]
print(f'Number of classes: {len(CLASS_NAMES)}')
print(f'Classes: {CLASS_NAMES}')

## 1. Count Images per Class (Raw Data)

In [ ]:
raw_dir = Path('../data/raw/animal-image-classification')
class_counts = {}

if raw_dir.exists():
    for cls in CLASS_NAMES:
        cls_dir = raw_dir / cls
        if cls_dir.exists():
            n = len([f for f in cls_dir.iterdir() if f.suffix.lower() in {'.jpg','.jpeg','.png'}])
            class_counts[cls] = n
            print(f'  {cls:<12} {n:>5} images')
        else:
            print(f'  {cls:<12}  NOT FOUND')
    print(f'\nTotal: {sum(class_counts.values())} images')
else:
    print('Raw data not found. Run scripts/download_data.py first.')
    # Approximate placeholder counts
    class_counts = {cls: 120 for cls in CLASS_NAMES}

## 2. Class Distribution Plot

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
classes = list(class_counts.keys())
counts  = list(class_counts.values())
colors  = sns.color_palette('husl', len(classes))

bars = ax.bar(classes, counts, color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Animal Image Classification Dataset — Images per Class',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Animal Class')
ax.set_ylabel('Number of Images')
plt.xticks(rotation=45, ha='right')

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(count), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
Path('../outputs/figures').mkdir(parents=True, exist_ok=True)
plt.savefig('../outputs/figures/class_distribution_raw.png', dpi=150, bbox_inches='tight')
plt.show()

if len(counts) > 1:
    print(f'Max class: {classes[counts.index(max(counts))]} ({max(counts)} images)')
    print(f'Min class: {classes[counts.index(min(counts))]} ({min(counts)} images)')
    print(f'Imbalance ratio: {max(counts)/min(counts):.2f}x')

## 3. Sample Images per Class

In [ ]:
raw_dir = Path('../data/raw/animal-image-classification')

if raw_dir.exists():
    cols = 5
    rows = 3  # 15 classes in a 5×3 grid
    fig, axes = plt.subplots(rows, cols, figsize=(16, 10))
    axes = axes.flatten()

    for i, cls in enumerate(CLASS_NAMES):
        cls_dir = raw_dir / cls
        if cls_dir.exists():
            imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.jpeg')) + list(cls_dir.glob('*.png'))
            if imgs:
                img = Image.open(imgs[0]).convert('RGB')
                axes[i].imshow(img)
                axes[i].set_title(cls.capitalize(), fontsize=11, fontweight='bold')
                axes[i].axis('off')
        else:
            axes[i].text(0.5, 0.5, f'{cls}\n(not found)', ha='center', va='center',
                         transform=axes[i].transAxes, color='red')
            axes[i].axis('off')

    plt.suptitle('Sample Images — 15 Animal Classes', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/figures/sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Raw data not found. Download the dataset first.')

## 4. Image Size Distribution

In [ ]:
raw_dir = Path('../data/raw/animal-image-classification')

if raw_dir.exists():
    widths, heights, aspect_ratios = [], [], []

    for img_path in list(raw_dir.rglob('*.jpg'))[:500]:
        try:
            with Image.open(img_path) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)
                aspect_ratios.append(w / h)
        except:
            pass

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(widths, bins=30, color='steelblue', edgecolor='white')
    axes[0].axvline(224, color='red', linestyle='--', label='Target: 224px')
    axes[0].set_title('Width Distribution')
    axes[0].set_xlabel('Width (px)')
    axes[0].legend()

    axes[1].hist(heights, bins=30, color='coral', edgecolor='white')
    axes[1].axvline(224, color='red', linestyle='--', label='Target: 224px')
    axes[1].set_title('Height Distribution')
    axes[1].set_xlabel('Height (px)')
    axes[1].legend()

    axes[2].hist(aspect_ratios, bins=30, color='mediumseagreen', edgecolor='white')
    axes[2].axvline(1.0, color='red', linestyle='--', label='Square (1:1)')
    axes[2].set_title('Aspect Ratio Distribution')
    axes[2].set_xlabel('Width / Height')
    axes[2].legend()

    plt.suptitle(f'Image Size Distribution (n={len(widths)} samples)', fontsize=13)
    plt.tight_layout()
    plt.show()

    print(f'Width  — mean: {np.mean(widths):.0f}px  median: {np.median(widths):.0f}px  min: {min(widths)}  max: {max(widths)}')
    print(f'Height — mean: {np.mean(heights):.0f}px  median: {np.median(heights):.0f}px  min: {min(heights)}  max: {max(heights)}')
else:
    print('Raw data not found.')

## 5. Load Class Metadata

In [ ]:
with open('../data/metadata/class_names.json') as f:
    meta = json.load(f)

print(f'Num classes: {meta["num_classes"]}')
print('\nClass → Index mapping:')
for cls, idx in meta['class_to_index'].items():
    print(f'  {idx:>2}  {cls}')